# Liu et al. (2026): the claims whose data are elsewhere

Paper: doi:10.1038/s41556-026-02009-4. The registry's dataset is DANDI:001752, version 0.260825.0222
(`reproduce_liu_2026.ipynb`); this notebook tests claims that the claims file marks `data: elsewhere`,
on the data its `where` names, and only those cheap enough for this run. They are counted apart from the
dataset's claims.

Sources, all pinned:
- the paper's code and per-cell tables, github.com/RFLiu2021/AIC_proj at commit
  44cd8a6dfbb1f1ce363d73742e4f3652086b0f9d (`aic_tables.CODE_COMMIT`);
- DANDI:001750 (somatic Na+ currents in outside-out patches), version 0.260825.0222, its only published
  version, streamed;
- DANDI:001746 (Patch-seq electrophysiology), version 0.260825.0222, its only published version, each file
  read into memory in one request.

| claim | source | here |
|---|---|---|
| ven-map-et-ct | mappingResults_neurons.csv (and results_Ext_240415.csv) | tested |
| patchseq-mapping | the same two tables | tested |
| fig3e-phase-peak-index | PeakIdx_Fig5e.xlsx | tested |
| fig4j-eepsp-amplitude | sumData_StimWM.xlsx | tested |
| fig5-connectivity-total, fig5-connectivity-by-type | metadata_multiPatch.xlsx, the code's annotation of DANDI:001751 | tested |
| fig4d-sodium-current | DANDI:001750 | tested |
| fig8a-metabolite-gradient, fig8de-glycolytic-bias | Metabolic_data.xlsx, the code's table of the MetaboLights data | tested |
| fig2d-f-l23-it-tau-width, ven-subtypes-similar-physiology | DANDI:001746 (Patch-seq electrophysiology), with the code's cell table | tested |
| the other elsewhere claims | | blocked; see the attempt record |

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt

import aic_tables as T

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)
print("code commit:", T.CODE_COMMIT, "; sodium dandiset:", T.SODIUM_DANDISET, T.SODIUM_VERSION)

In [ ]:
# What each table holds.
mapping = T.csv("patchSeqMappingTo10X/data/mappingRes/mappingResults_neurons.csv").rename(columns={"Unnamed: 0": "cell"})
mapping_ext = T.csv("patchSeqMappingTo10X/data/mappingRes/results_Ext_240415.csv").rename(columns={"Unnamed: 0": "cell"})
peak = T.excel("ephysio/data_ephy/PeakIdx_Fig5e.xlsx", "Sheet1")
stim = T.excel("ephysio/data_ephy/sumData_StimWM.xlsx", "re-orgnized_data")
multi = T.excel("ephysio/data_ephy/rawdata_ephysio/metadata_multiPatch.xlsx", "metadata_multiPatch", header=0)
na_meta = T.csv("ephysio/data_ephy/rawdata_ephysio/metadata_outSideOutSodiumCurrentRec.csv")
for name, df in [("mappingResults_neurons", mapping), ("results_Ext_240415", mapping_ext), ("PeakIdx_Fig5e", peak),
                 ("sumData_StimWM", stim), ("metadata_multiPatch", multi), ("metadata_outSideOutSodiumCurrentRec", na_meta)]:
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns; columns: {list(df.columns) if df.columns.dtype == object else list(df.iloc[0])}")
print("\npatchCellTypeIDs x patchCellSubtypeNames (mappingResults_neurons):")
print(pd.crosstab(mapping.patchCellSubtypeNames.fillna("(none)"), mapping.patchCellTypeIDs).to_string())

## ven-map-et-ct

> The 29 morphologically identified VENs do not map to one molecular cluster; 14 map to L5 ET and 15 to
> L5/6 CT.

VENs are the cells whose subtype name contains "VEN". The cluster numbers of L5 ET (16) and L5/6 CT (8) are
the code's (`morpho/Fig3f_plotCellTypesAIS_Cmp.m`, `cluster_VEN_L5ET = {16}`, `cluster_VEN_L56CT = {8}`).
The claim names mappingResults_neurons.csv; the excitatory table results_Ext_240415.csv in the same folder
gives other subtype names and numbers, and is shown too.

In [ ]:
for name, df in [("mappingResults_neurons.csv", mapping), ("results_Ext_240415.csv", mapping_ext)]:
    v = df[df.patchCellSubtypeNames.fillna("").str.contains("VEN", case=False)]
    print(f"{name}: {len(v)} VENs; subtype x cluster:")
    print(pd.crosstab(v.patchCellSubtypeNames, v.predictClustersIDs).to_string())
    print(f"  L5 ET (cluster {T.CLUSTER_L5_ET}): {(v.predictClustersIDs == T.CLUSTER_L5_ET).sum()}; "
          f"L5/6 CT (cluster {T.CLUSTER_L56_CT}): {(v.predictClustersIDs == T.CLUSTER_L56_CT).sum()}; "
          f"other clusters: {(~v.predictClustersIDs.isin([T.CLUSTER_L5_ET, T.CLUSTER_L56_CT])).sum()}\n")

**Result: reproduced.** In mappingResults_neurons.csv, the file the claim names, 30 cells are VENs; 15 map
to L5 ET (cluster 16), 14 to L5/6 CT (cluster 8) and 1 to cluster 14, so the VENs do not map to one cluster
and both counts are within their tolerances (14 +- 1.4, 15 +- 1.5). In results_Ext_240415.csv the 29 VENs
split exactly as the paper says, 14 VEN-Long in cluster 16 and 15 VEN-Short in cluster 8. These are the
code's mapping results, not a new mapping of the Patch-seq reads.

## patchseq-mapping

> Of 307 Patch-seq neurons with transcriptomes, 273 excitatory neurons map to 18 of the 20 excitatory
> types and 34 inhibitory neurons to 17 of the 33 inhibitory types.

Counts from the code's mapping tables. Excitatory neurons are the 273 rows of results_Ext_240415.csv (the
excitatory mapping) and inhibitory neurons the rows of mappingResults_neurons.csv not in it; the cells the
code labels as interneurons (`patchCellTypeIDs` 0) are a second reading.

In [ ]:
print("mappingResults_neurons.csv rows:", len(mapping), "; results_Ext_240415.csv rows:", len(mapping_ext))
ext_types = mapping_ext.predictClustersIDs.nunique()
print("excitatory: neurons", len(mapping_ext), "; distinct clusters", ext_types,
      "; cluster numbers", sorted(mapping_ext.predictClustersIDs.unique()))
inh_a = mapping[~mapping.cell.isin(mapping_ext.cell)]
inh_b = mapping[mapping.patchCellTypeIDs == 0]
for label, inh in [("rows not in the excitatory table", inh_a), ("patchCellTypeIDs 0 (interneurons)", inh_b)]:
    print(f"inhibitory, {label}: neurons {len(inh)}; distinct clusters {inh.predictClustersIDs.nunique()}; "
          f"cluster numbers {sorted(inh.predictClustersIDs.unique())}")
print("excitatory-table cells not in mappingResults_neurons.csv:", sorted(set(mapping_ext.cell) - set(mapping.cell)))

**Result: reproduced (resting on an assumption).** Excitatory: 273 neurons (results_Ext_240415.csv) in
17 distinct clusters, within the tolerances of 273 and 18 (+- 1.8). Inhibitory, taking the cells the code
labels as interneurons (`patchCellTypeIDs` 0, the assumption): 33 neurons in 17 distinct clusters, within
the tolerances of 34 (+- 3.4) and 17 (+- 1.7). The other reading, every row outside the excitatory table,
gives 35 neurons in 19 clusters, outside the tolerance of 17 types. The tables do not say which numbering
the inhibitory cluster numbers follow, so the type counts rest on reading them as inhibitory types.

## fig3e-phase-peak-index

> VENs have lower AP phase-plane peak indices than every PC type.

Values as the code reads them (`ephysio/Fig3e_plotBoxplot_PeakIdx_cellTypes.m`: rows 1 to 30, 22, 122, 29,
17 and 13 of the six columns). Kruskal-Wallis, then Dunn's test with Bonferroni correction (Fig. 3e legend;
the sheet's "statistics" names "Dunn's Test Results (Bonferroni-corrected p-values)").

In [ ]:
n_code = {"L23IT_HAPCAL1": 30, "ITrorb": 22, "L5IT": 122, "VEN": 29, "L5ET": 17, "L6PC": 13}
groups = {}
for c in range(0, peak.shape[1], 3):
    name = peak.iat[0, c]
    vals = pd.to_numeric(peak.iloc[1:, c + 1], errors="coerce").to_numpy()
    groups[name] = vals[: n_code[name]]
    print(f"{name}: code n {n_code[name]}, non-missing in the code's rows {np.isfinite(groups[name]).sum()}, "
          f"values in the column {np.isfinite(vals).sum()}; median {np.nanmedian(groups[name]):.3f}, "
          f"range {np.nanmin(groups[name]):.3f} to {np.nanmax(groups[name]):.3f}")
groups = {k: v[np.isfinite(v)] for k, v in groups.items()}
kw = stats.kruskal(*groups.values())
print(f"\nKruskal-Wallis H = {kw.statistic:.2f}, p = {kw.pvalue:.3g}")
dunn = T.dunn_bonferroni(groups)
print(dunn[(dunn.a == "VEN") | (dunn.b == "VEN")].round(5).to_string())

**Result: reproduced.** The VEN median (0.117) is the lowest of the six groups (the others 0.223 to 0.338);
Kruskal-Wallis H = 42.4, p = 4.9e-08; Dunn's test with Bonferroni correction separates VENs from every PC
type (adjusted p 0.001 for L2/3 IT HPCAL1, 2e-05 for IT RORB, below 1e-05 for L5 IT, 7e-05 for L5 ET, 3e-05
for L6 PC). The values are the code's per-cell indices; the code's group sizes (30, 22, 122, 29, 17, 13)
differ from the Fig. 2 legend's, and one IT RORB and one L5 IT row are empty.

## fig4j-eepsp-amplitude

> eEPSP amplitudes are higher in VENs than in adjacent PCs, with effects of cell type and stimulation level
> in an ANOVA.

The sheet "re-orgnized_data" (columns stim current, EPSP_amp_PC, EPSP_amp_VEN, rows 2 to 79: 26 pairs at
three currents, the rows the code averages from; its rows 81 to 83 hold the code's means). Two-way ANOVA
of amplitude on cell type and stimulation level (criterion); with their interaction and with pair as a
blocking factor as variants. Amplitudes are in V in the sheet (the code multiplies by 1000 for mV).

In [ ]:
block = stim.iloc[1:79, 6:9].apply(pd.to_numeric, errors="coerce")
block.columns = ["current_mA", "PC", "VEN"]
print("rows:", len(block), "; per current:", block.current_mA.value_counts().sort_index().to_dict(),
      "; missing:", int(block.isna().sum().sum()))
block["pair"] = block.groupby("current_mA").cumcount()
long = block.melt(id_vars=["current_mA", "pair"], value_vars=["PC", "VEN"], var_name="cell", value_name="amp")
long["amp_mV"] = long.amp * 1000
print("means (mV) by current and cell type:")
print(long.pivot_table(index="current_mA", columns="cell", values="amp_mV", aggfunc="mean").round(2).to_string())
print("code's means (rows 81 to 83, x1000):", (stim.iloc[81:84, 7:9].astype(float) * 1000).round(2).values.tolist())
fit = smf.ols("amp_mV ~ C(cell) + C(current_mA)", data=long).fit()
print("\ntwo-way ANOVA, main effects:"); print(sm.stats.anova_lm(fit, typ=2).round(5).to_string())
fit2 = smf.ols("amp_mV ~ C(cell) * C(current_mA)", data=long).fit()
print("\nwith interaction:"); print(sm.stats.anova_lm(fit2, typ=2).round(5).to_string())
fit3 = smf.ols("amp_mV ~ C(cell) + C(current_mA) + C(pair)", data=long).fit()
print("\nwith pair as a block (variant):"); print(sm.stats.anova_lm(fit3, typ=2).loc[["C(cell)", "C(current_mA)"]].round(5).to_string())
d = block.VEN - block.PC
print(f"\nVEN minus PC per pair and current: median {1000*d.median():.2f} mV, VEN higher in {(d > 0).sum()} of {d.notna().sum()}")

**Result: reproduced.** VEN amplitudes are higher at each current (5.01 against 3.97, 10.57 against 5.84 and
14.54 against 8.69 mV, the code's own means), and in the two-way ANOVA the cell-type effect has p = 0.0034
(the paper's value) and the stimulation level p = 8e-05 (paper < 0.001). With the interaction (p = 0.29 for
it) or with pair as a block, cell type stays significant (p = 0.0033, 0.00017). The values are the code's
per-pair amplitudes, not a new measurement from raw recordings.

## fig5-connectivity-total and fig5-connectivity-by-type

> Of 1,292 potential connections among 377 layer 5b neurons, 31 monosynaptic connections were found.
> PCs connect to PCs in 24 of 503 pairs, VENs make no connections to VENs (0 of 110) or PCs (0 of 200), PCs
> connect to VENs in 1 of 200, PCs and INs connect in 2 and 4 of 65, and no VEN-IN connections occur.

The claims ask for the deposit's own connection annotations. DANDI:001751 is 7.0 GB of 301 files; the
paper's code holds its annotation, `metadata_multiPatch.xlsx`, one row per tested pair (presynaptic and
postsynaptic channel and cell type, and, for a connection, its type and quality), with the NWB file of each
presynaptic cell. That table is used. "GABA" is read as IN.

In [ ]:
pairs = multi.dropna(subset=["preSynCellType", "postSynCellType"]).copy()
pairs["file"] = pairs.filename.ffill()
pairs["pre_ch"] = pairs.preSynChannel.ffill()
pairs["connected"] = pairs.ConnType.notna()
print("tested pairs (rows with both cell types):", len(pairs), "; connections:", int(pairs.connected.sum()),
      "; presynaptic files:", pairs.file.nunique(), "; subjects:", multi.Subject_id.nunique())
print("connections by type and quality:", pairs[pairs.connected].groupby(["ConnType", "connQuality"]).size().to_dict())
print("rows naming a file:", int(multi.filename.notna().sum()), "; rows without a presynaptic channel after filling:",
      int(pairs.pre_ch.isna().sum()))

**Result: partially reproduced.** The annotation lists 1,416 tested pairs, within the tolerance of 1,292
(+- 129), and 26 connections (14 excitatory, 12 inhibitory), outside the tolerance of 31 (+- 3.1). The
number of neurons cannot be checked from it, as a neuron is named only by a channel of a recording.

In [ ]:
pairs["pre"] = pairs.preSynCellType.replace({"GABA": "IN"})
pairs["post"] = pairs.postSynCellType.replace({"GABA": "IN"})
tab = pairs.groupby(["pre", "post"]).agg(tested=("connected", "size"), connected=("connected", "sum"))
tab["paper_tested"] = pd.Series({("PC", "PC"): 503, ("VEN", "VEN"): 110, ("VEN", "PC"): 200, ("PC", "VEN"): 200,
                                 ("PC", "IN"): 65, ("IN", "PC"): 65, ("VEN", "IN"): 55, ("IN", "VEN"): 55})
tab["paper_connected"] = pd.Series({("PC", "PC"): 24, ("VEN", "VEN"): 0, ("VEN", "PC"): 0, ("PC", "VEN"): 1,
                                    ("PC", "IN"): 2, ("IN", "PC"): 4, ("VEN", "IN"): 0, ("IN", "VEN"): 0})
print(tab.to_string())
print("\nconnections listed:")
print(pairs[pairs.connected][["file", "pre_ch", "posySynChannel", "pre", "post", "ConnType", "connQuality", "projection"]].to_string())

**Result: partially reproduced.** Six of the eight connection counts match the paper: VEN-to-VEN 0, PC-to-VEN
1, PC-to-IN 2, IN-to-PC 4, VEN-to-IN 0 and IN-to-VEN 0. Two do not: PC-to-PC is 18 (paper 24 +- 2.4) and
VEN-to-PC is 1 (paper 0), a connection annotated "Ext, Excellent" from channel 4 to 6 of
sub-Mfa190314_ses-190315s2_obj-1e6g6jn. The tested-pair counts differ from the paper's for every type
(PC-to-PC 749 against 503, VEN-to-VEN 59 against 110). The table may not be the annotation the paper counted
from; this checks the deposited annotation, not the recordings.

## fig4d-sodium-current

> Somatic Na+ currents in outside-out patches are lower in VENs (n = 14) than in L5 PCs (n = 21).

Per patch, the sweeps of each command step are averaged, the current is referred to the mean over 2 to
0.2 ms before the step and smoothed over 0.1 ms (10 samples at 100 kHz), and its most negative value in 0.1
to 3 ms after the step onset is the peak inward current (standard choices; the Methods give the steps,
-80 to 0 mV in 5 mV for 25 ms, and leak subtraction online). The patch's Na+ current is the largest peak
over the steps from -60 to 0 mV. Reading (b), current density, divides by the patch area estimated from
capacitance; the files hold no capacitance, and the Methods give one area (~5 um2 from ~50 fF) and say
similar areas were sampled, so (b) is (a) divided by a constant and gives the same test. Cell types come
from the code's metadata_outSideOutSodiumCurrentRec.csv, which also gives a `max_Na_current` per cell,
used as a check.

In [ ]:
na_assets = T.sodium_assets()
na_files = T.sodium_all(na_assets)
rows = []
for f in na_files:
    by_step = {}
    for name, s in f["sweeps"].items():
        v, i = s["command_mV"], s["current_pA"]
        onset = np.flatnonzero(np.abs(v - v[0]) > 1)
        if len(onset) == 0:
            continue
        step = round(float(v[onset[0] + 5]))
        by_step.setdefault(step, []).append((onset[0], i, s["rate"]))
    best, best_step = 0.0, None
    for step, sw in by_step.items():
        if not -60 <= step <= 0:
            continue
        on, rate = sw[0][0], sw[0][2]
        mean_i = np.mean([x[1] for x in sw], axis=0)
        sm_i = np.convolve(mean_i, np.ones(10) / 10, mode="same")
        b = sm_i[on - int(0.002 * rate): on - int(0.0002 * rate)].mean()
        pk = sm_i[on + int(0.0001 * rate): on + int(0.003 * rate)].min() - b
        if pk < best:
            best, best_step = pk, step
    rows.append(dict(file=f["file"], cell_id=f["cell_id"], subject=f["subject"], n_sweeps=len(f["sweeps"]),
                     steps=f"{min(by_step)} to {max(by_step)}", onset_ms=on / rate * 1000, peak_pA=best, at_mV=best_step))
na = pd.DataFrame(rows).merge(na_meta, left_on="file", right_on="fileName_nwb", how="left")
print("files:", len(na), "; with a cell type:", int(na.cellType.notna().sum()), "; by type:", na.cellType.value_counts().to_dict())
print("sampling rates (Hz):", sorted({round(s['rate']) for f in na_files for s in f['sweeps'].values()}),
      "; steps per file:", na.steps.value_counts().to_dict(), "; step onset (ms):", sorted(na.onset_ms.round(2).unique()))
print(na[["file", "cell_id", "cellType", "n_sweeps", "peak_pA", "at_mV", "max_Na_current"]].round(2).to_string())
r = stats.spearmanr(na.peak_pA, na.max_Na_current)
print(f"\nagreement with the code's max_Na_current: Spearman rho = {r[0]:.2f}, p = {r[1]:.2g}; "
      f"median ratio {np.median(na.peak_pA / na.max_Na_current):.2f}")

In [ ]:
for col, label in [("peak_pA", "peak inward current here"), ("max_Na_current", "the code's max_Na_current")]:
    v, p = na[na.cellType == "VEN"][col].abs(), na[na.cellType == "PC"][col].abs()
    t = stats.mannwhitneyu(v, p, alternative="two-sided")
    print(f"{label}: |I| VEN median {v.median():.1f} (n {len(v)}, {v.min():.1f} to {v.max():.1f}) against PC "
          f"{p.median():.1f} pA (n {len(p)}, {p.min():.1f} to {p.max():.1f}); U = {t.statistic:.0f}, p = {t.pvalue:.4f}")
area_um2 = 5.0
v, p = na[na.cellType == "VEN"].peak_pA.abs() / area_um2, na[na.cellType == "PC"].peak_pA.abs() / area_um2
print(f"reading (b), density at the Methods' ~5 um2 for every patch: VEN {v.median():.2f} against PC {p.median():.2f} pA/um2, "
      f"p = {stats.mannwhitneyu(v, p, alternative='two-sided').pvalue:.4f}")
print("by session:")
na["session"] = na.file.str.extract(r"sess-(\w+?)_obj")[0]
print(na.groupby(["session", "cellType"]).peak_pA.agg(["size", "median"]).round(1).to_string())

**Result: reproduced.** The peak inward current is smaller in VENs (median |I| 23.0 pA, n = 14) than in PCs
(34.6 pA, n = 21), two-tailed rank-sum p = 0.0023; the density reading gives the same p, and VENs are
smaller in both sessions. The measure follows the code's per-cell max_Na_current closely (rho = 0.97, a
median 1.26 times larger in magnitude), which gives p = 0.0010.

## fig8a-metabolite-gradient

> Of 581 metabolites, 228 differ between AIC and V1 and only 108 between AIC and parietal cortex.

The claim names MetaboLights MTBLS13927; the paper's code holds the normalized table it analysed,
`VEN_Metabolism/Metabolic_data.xlsx` (sheet "Identified"; its notebook reads this table), which is used here.
Per metabolite, Welch's ANOVA over the three regions, then the Games-Howell test (Fig. 8 legend), at
P < 0.05 (criterion). Readings: a metabolite differs when its Games-Howell p < 0.05 (a) alone or (b) only
after a Welch ANOVA p < 0.05; values raw, as the code leaves its log10 line commented out, with log10 as a
variant.

In [ ]:
met = T.excel("VEN_Metabolism/Metabolic_data.xlsx", "Identified", header=0)
cols = {g: [c for c in met.columns if str(c).startswith(g + "_")] for g in ["AIC", "PrC", "V1"]}
print("metabolites:", len(met), "; distinct names:", met["Metabolite Name"].nunique(), "; samples:",
      {g: c for g, c in cols.items()}, "; zero values:", int((met[sum(cols.values(), [])] == 0).sum().sum()))
rows = []
for transform in ("raw", "log10"):
    for _, r in met.iterrows():
        g = {k: r[c].astype(float).to_numpy() for k, c in cols.items()}
        if transform == "log10":
            g = {k: np.log10(v + 1e-9) for k, v in g.items()}
        f, p = T.welch_anova(*g.values())
        rows.append(dict(transform=transform, name=r["Metabolite Name"], p_welch=p,
                         gh_AIC_V1=T.games_howell(g["AIC"], g["V1"], 3), gh_AIC_PrC=T.games_howell(g["AIC"], g["PrC"], 3),
                         AIC=g["AIC"].mean(), PrC=g["PrC"].mean(), V1=g["V1"].mean()))
mres = pd.DataFrame(rows)
out = []
for transform, g in mres.groupby("transform"):
    for reading, gate in (("(a) Games-Howell alone", np.ones(len(g), bool)), ("(b) after Welch p < 0.05", g.p_welch < 0.05)):
        out.append(dict(transform=transform, reading=reading, AIC_V1=int(((g.gh_AIC_V1 < 0.05) & gate).sum()),
                        AIC_PrC=int(((g.gh_AIC_PrC < 0.05) & gate).sum()), untestable=int(g.p_welch.isna().sum())))
print(pd.DataFrame(out).to_string())

**Result: partially reproduced.** The gradient holds: about twice as many metabolites differ between AIC and
V1 as between AIC and PrC under every reading. The counts are lower than the paper's: raw values give 179
and 85 (Games-Howell alone) or 175 and 84 (after Welch), against 228 (+- 23) and 108 (+- 11); log10 values
give 194 and 98 (the second within tolerance) or 190 and 95. The table holds 558 identified metabolites, where the paper
reports 581.

## fig8de-glycolytic-bias

> Compared with V1, AIC shows elevated citrate, reduced 2-ketobutyrate and increased glutamate, with altered
> expression of oxidative enzyme genes, read as a glycolytic bias.

The three metabolites the criterion names, by their names in the table ("Citric acid", "2-Ketobutyric
acid", "L-Glutamic acid"), Games-Howell AIC against V1 after Welch's ANOVA, raw values (log10 shown).

In [ ]:
named = ["Citric acid", "2-Ketobutyric acid", "L-Glutamic acid"]
print(mres[mres.name.isin(named)][["transform", "name", "AIC", "PrC", "V1", "p_welch", "gh_AIC_V1"]]
      .sort_values(["name", "transform"]).to_string())

**Result: partially reproduced.** Citrate is higher in AIC than in V1 (Games-Howell p = 0.0005, Welch
p = 0.0008) and glutamate (L-glutamic acid) is higher (p = 0.0001). 2-Ketobutyrate is lower in AIC (mean
24.3 against 66.5) but not significantly (Games-Howell p = 0.18, Welch p = 0.17; on log10 values p = 0.086). The gene-expression part
of the statement is not tested here.

## fig2d-f-l23-it-tau-width

> Among projection neuron subclasses, L2/3 IT HPCAL1 neurons have a small membrane time constant and a
> narrow AP width.

From DANDI:001746 (version 0.260825.0222), streamed. Each file is read into memory in one request (helper
`aic_tables.patchseq_features`). Per cell, as the Methods define them: tau from an exponential fitted from
the stimulation onset to the bottom of the sag on each hyperpolarizing sweep, averaged over the sweeps; the
first AP of the rheobase sweep, its threshold at the largest third derivative in the 20 ms before the peak,
its half-width at half the amplitude above threshold. APs are found by a crossing of 0 mV and failed fits
(tau outside 0 to 600 ms) are dropped, standard choices. Subclasses as the code's Fig. 2d script groups the
predicted clusters (`ephysio/Fig2D_ephyFeatCmp.m`), with the code's cell table
(`metaInfo_ephysio_patchSeq.csv`); the script also drops tau values of 1 ms or less, as here. Two-tailed
rank-sum tests between every pair of subclasses, Benjamini-Hochberg over the 15 pairs of each feature
(the family is not stated; a standard choice).

In [ ]:
from statsmodels.stats.multitest import multipletests
ps_assets = T.patchseq_assets()
ps_meta = T.csv("ephysio/data_ephy/rawdata_ephysio/metaInfo_ephysio_patchSeq.csv")
ps_meta["subclass"] = [T.subclass(c, t) for c, t in zip(ps_meta.predictCluster, ps_meta.cellType)]
byname = {a["path"].split("/")[-1]: a for a in ps_assets}
print("dandiset files:", len(ps_assets), "; rows of the code's table:", len(ps_meta),
      "; table rows naming a dandiset file:", int(ps_meta.filename_nwb.isin(byname).sum()))
print("cells by subclass (code's grouping):", ps_meta.subclass.value_counts(dropna=False).to_dict())
sel = ps_meta[ps_meta.subclass.notna() | ps_meta.cellSubtype.astype(str).str.contains("VEN", case=False)]
feats = pd.DataFrame(T.patchseq_all([byname[f] for f in sel.filename_nwb if f in byname]))
ps = sel.merge(feats, left_on="filename_nwb", right_on="file", how="left")
print("cells read:", len(feats), "; errors:", int(feats.get("error", pd.Series(dtype=object)).notna().sum()),
      "; sampling rates:", feats.rate.value_counts().to_dict(), "; step lengths (ms):", feats.step_ms.round(1).value_counts().to_dict())
print("hyperpolarizing sweeps fitted per cell: median", ps.n_tau_sweeps.median(), "; failed fits in all:", int(ps.n_tau_failed.sum()),
      "; cells with no AP found:", int(ps.half_width_ms.isna().sum()))
print(ps[["tau_ms", "rheobase_pA", "threshold_mV", "amplitude_mV", "half_width_ms"]].describe().round(2).to_string())

In [ ]:
order = ["L2/3 IT HPCAL1", "IT RORB", "L5 IT FEZF2", "VEN", "L5 ET", "L6 PC"]
g6 = ps[ps.subclass.isin(order)]
for feat, keep in (("tau_ms", lambda x: x > 1), ("half_width_ms", lambda x: x > 0)):
    vals = {k: g6[g6.subclass == k][feat].dropna() for k in order}
    vals = {k: v[keep(v)] for k, v in vals.items()}
    print(f"\n{feat}: median (n) by subclass:", {k: (round(v.median(), 2), len(v)) for k, v in vals.items()})
    pairs = [(a, b) for i, a in enumerate(order) for b in order[i + 1:]]
    p = [stats.mannwhitneyu(vals[a], vals[b], alternative="two-sided").pvalue for a, b in pairs]
    q = multipletests(p, method="fdr_bh")[1]
    res = pd.DataFrame(dict(a=[x[0] for x in pairs], b=[x[1] for x in pairs], p=p, fdr=q))
    print(res[(res.a == "L2/3 IT HPCAL1") | (res.b == "L2/3 IT HPCAL1")].round(5).to_string())
    print("lowest median:", min(vals, key=lambda k: vals[k].median()))
print("\nby subject, median tau (ms) and half-width (ms), L2/3 IT HPCAL1 against the other five subclasses pooled:")
g6 = g6.assign(group=np.where(g6.subclass == "L2/3 IT HPCAL1", "L2/3 IT HPCAL1", "others"))
print(g6[g6.tau_ms > 1].groupby(["subject_id", "group"]).agg(n=("tau_ms", "size"), tau=("tau_ms", "median"),
      width=("half_width_ms", "median")).unstack("group").round(2).to_string())

**Result: not reproduced.** Under neither reading does the claim hold for AP width. L2/3 IT HPCAL1 has the
lowest median tau (31.8 ms, n = 30; others 38.2 to 54.1 ms), but no pairwise difference with it survives
FDR (0.064 to 0.82). Its median AP half-width (2.10 ms) is not the lowest; IT RORB is lower (1.92 ms), and
L2/3 IT HPCAL1 is narrower than L5 IT FEZF2 and VEN only (FDR 8e-05 each), not than IT RORB, L5 ET or L6 PC
(FDR 0.62 to 0.78). The group sizes by the code's grouping (30, 21, 125, 29, 18, 14) differ from the
legend's (30, 17, 119, 22, 16, 13), and the features are this attempt's extraction, not the paper's
PatchElectroLab values; some AP detections are implausible (half-widths 0.08 to 12 ms, amplitudes down to
0.7 mV), so the outcome may be a failure of this attempt.

## ven-subtypes-similar-physiology

> VEN-L and VEN-S have similar physiological properties.

A claim of absence, judged on two-tailed rank-sum tests with Benjamini-Hochberg FDR over AP width and
amplitude (Extended Data Fig. 9c,d). DANDI:001746 holds the Patch-seq VENs only; the code's table names
them VEN_Long and VEN_Short (or VEN_short), far fewer than the paper's 85 VEN-L and 41 VEN-S.

In [ ]:
ven = ps[ps.cellSubtype.astype(str).str.contains("VEN", case=False)].copy()
ven["sub"] = np.where(ven.cellSubtype.str.lower().str.contains("long"), "VEN-L", "VEN-S")
print("VENs by subtype:", ven["sub"].value_counts().to_dict())
pv = []
for feat in ("half_width_ms", "amplitude_mV"):
    a, b = ven[ven["sub"] == "VEN-L"][feat].dropna(), ven[ven["sub"] == "VEN-S"][feat].dropna()
    t = stats.mannwhitneyu(a, b, alternative="two-sided")
    pv.append(t.pvalue)
    print(f"{feat}: VEN-L median {a.median():.2f} (n {len(a)}) against VEN-S {b.median():.2f} (n {len(b)}), p = {t.pvalue:.4f}")
print("FDR (BH over the two):", np.round(multipletests(pv, method="fdr_bh")[1], 4))

**Result: reproduced.** Neither difference is significant after FDR: AP half-width VEN-L 3.33 ms (n = 18)
against VEN-S 2.76 ms (n = 16), p = 0.042, FDR 0.084; amplitude 88.1 against 91.4 mV, p = 0.10, FDR 0.10.
The dandiset holds 34 of the paper's 126 VENs, so the test has less power than the paper's.

## Summary

| claim | outcome |
|---|---|
| ven-map-et-ct | reproduced |
| patchseq-mapping | reproduced, resting on an assumption (which rows are inhibitory) |
| fig3e-phase-peak-index | reproduced |
| fig4j-eepsp-amplitude | reproduced |
| fig5-connectivity-total | partially reproduced |
| fig5-connectivity-by-type | partially reproduced |
| fig4d-sodium-current | reproduced |
| fig8a-metabolite-gradient | partially reproduced |
| fig8de-glycolytic-bias | partially reproduced |
| fig2d-f-l23-it-tau-width | not reproduced |
| ven-subtypes-similar-physiology | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

All but fig4d-sodium-current, fig2d-f-l23-it-tau-width and ven-subtypes-similar-physiology are tested on
the code's per-cell tables, not on raw data: they check that the
published tables support the paper's numbers. The connection counts come from the code's annotation of
DANDI:001751, which may not be the one the paper counted.